# Техническая часть: сбор данных ФНС и геокодирование

Этот Notebook **не требуется запускать для просмотра анализа**. Его задача — документировать способы получения исходных данных для `Analysis_code.ipynb`.

Запросы к ФНС и Nominatim не отправляются автоматически: для повторного сбора нужно явно установить `RUN_COLLECTION = True` или `RUN_GEOCODING = True`. Соблюдайте ограничения сервисов, при HTTP 403/429 прекращайте массовые запросы.

In [1]:
import re
import time
from pathlib import Path

import pandas as pd
import requests

DATA_DIR = Path("../datasets")
DATA_DIR.mkdir(parents=True, exist_ok=True)
COMPANIES_PATH = DATA_DIR / "companies_checkpoint_1000.csv"
FINANCE_PATH = DATA_DIR / "financial_data_v2.csv"
GEOCODE_PATH = DATA_DIR / "geocode_cache.csv"

## 1. Функции обращения к ФНС

`get_org_id` получает ID по ИНН, `get_bfo_data` — годовые отчёты по ID, `extract_financial_data` — строки для таблицы «компания × год». При отсутствии показателя оставляем пропуск, а не заменяем его нулём.

In [2]:
session = requests.Session()
session.headers.update({
    "User-Agent": "ConstructionResearch/1.0",
    "Accept": "application/json",
    "Accept-Language": "ru-RU,ru;q=0.9",
})

def get_org_id(inn):
    url = "https://bo.nalog.gov.ru/advanced-search/organizations/search"
    response = session.get(url, params={"query": str(inn), "page": 0}, timeout=(10, 45))
    response.raise_for_status()
    organizations = response.json().get("content", [])
    for org in organizations:
        api_inn = re.sub(r"<[^>]+>", "", str(org.get("inn", ""))).strip()
        if api_inn == str(inn).strip():
            return org["id"]
    return None

def get_bfo_data(org_id):
    url = f"https://bo.nalog.gov.ru/nbo/organizations/{org_id}/bfo/"
    response = session.get(url, timeout=(10, 45))
    response.raise_for_status()
    return response.json()

In [3]:
def extract_financial_data(bfo_data, inn):
    records = []
    for report in bfo_data:
        corrections = report.get("typeCorrections") or []
        if not corrections:
            continue
        correction = corrections[0].get("correction") or {}
        financial = correction.get("financialResult")
        if not isinstance(financial, dict):
            continue

        before_tax = financial.get("current2300")
        interest = financial.get("current2330")
        ebit = before_tax + interest if before_tax is not None and interest is not None else None
        records.append({
            "inn": str(inn), "year": report["period"],
            "revenue": financial.get("current2110"),
            "profit_before_tax": before_tax,
            "interest_expense": interest,
            "ebit": ebit,
            "net_profit": financial.get("current2400"),
        })
    return records

## 2. Повторный сбор с контрольными точками (только по желанию)

Сначала загрузите готовый файл компаний с ИНН и `org_id`. Обработанные компании пропускаются. Результаты и статусы сохраняются при остановке. Эта процедура может занять значительное время и зависит от условий доступа ФНС.

**Важно:** это технический пример; его не следует запускать автоматически в итоговом аналитическом отчёте.

In [4]:
RUN_COLLECTION = False  # Поменяйте на True только если действительно нужен повторный сбор.

if RUN_COLLECTION:
    companies = pd.read_csv(COMPANIES_PATH, dtype=str)
    if "bfo_status" not in companies:
        companies["bfo_status"] = "pending"

    if FINANCE_PATH.exists():
        finance_records = pd.read_csv(FINANCE_PATH, dtype={"inn": str}).to_dict("records")
    else:
        finance_records = []

    def save_checkpoint():
        if finance_records:
            pd.DataFrame(finance_records).drop_duplicates(
                ["inn", "year"], keep="last"
            ).to_csv(FINANCE_PATH, index=False)
        companies.to_csv(COMPANIES_PATH, index=False)

    count = 0
    try:
        for idx, company in companies.iterrows():
            if company["bfo_status"] != "pending":
                continue
            inn, org_id = company["ИНН"], company["org_id"]
            try:
                if pd.isna(org_id):
                    org_id = get_org_id(inn)
                    if org_id is None:
                        companies.at[idx, "bfo_status"] = "needs_review"
                        continue
                    companies.at[idx, "org_id"] = str(org_id)
                    time.sleep(2)

                reports = get_bfo_data(org_id)
                rows = extract_financial_data(reports, inn)
                finance_records.extend(rows)
                companies.at[idx, "bfo_status"] = "success" if rows else "no_report"

            except requests.exceptions.HTTPError as error:
                companies.at[idx, "bfo_status"] = "error"
                status = error.response.status_code if error.response is not None else None
                print(f"HTTP {status}: {inn}")
                if status in (403, 429):
                    print("Ограничение доступа: сбор остановлен")
                    break
            except requests.exceptions.RequestException as error:
                companies.at[idx, "bfo_status"] = "error"
                print(f"Ошибка соединения: {inn}: {error}")
            except (KeyError, TypeError, ValueError) as error:
                companies.at[idx, "bfo_status"] = "parse_error"
                print(f"Ошибка данных: {inn}: {error}")
            finally:
                count += 1
                if count % 10 == 0:
                    save_checkpoint()
                    time.sleep(15)
                else:
                    time.sleep(2)
    finally:
        save_checkpoint()
        print("Сбор завершён/остановлен, прогресс сохранён")

## 3. Координаты мест регистрации

Кэш геокодирования `geocode_cache.csv` **уже используется в основном Notebook**, поэтому повторно обращаться к Nominatim не нужно. Важно проверять найденный регион: совпадение названия города само по себе недостаточно для точного позиционирования.

Если координаты отсутствуют, геокодируйте только **уникальные** комбинации города и региона с одним потоком, паузами и сохранением кэша. По правилам публичного Nominatim запросы должны быть редкими, а результаты — кэшироваться. Геокодирование юридического адреса по одним лишь городу/региону всегда приблизительно.

## Как использовать файлы

1. Для обычного анализа откройте **`Analysis_code.ipynb`** и нажмите **Restart & Run All**.
2. Файлы `Sample_41.20.csv`, `companies_checkpoint_1000.csv`, `financial_data_v2.csv`, `net_profit_data.csv`, `geocode_cache.csv` должны находиться в `datasets/`.
3. Этот Notebook запускайте только при необходимости повторного сбора данных.
4. Для построения карты в аналитическом Notebook требуются `geopandas`, `folium`, `mapclassify`.